# O mesmo agente em LangGraph: o que o framework automatiza

Este notebook reimplementa **exatamente o mesmo agente** do projeto anterior (harness do zero: calculadora + clima) usando LangGraph, para comparar lado a lado o que escrevemos a mao versus o que o framework assume por nos.

**Requer**: `pip install langgraph langchain-anthropic` e `ANTHROPIC_API_KEY` configurada. Como no notebook anterior, o codigo de execucao real (que chama a API) nao pode ser testado neste ambiente sem internet -- mas a estrutura segue os padroes atuais e documentados do LangGraph.

**Pre-requisito**: ter feito o notebook "harness do zero" primeiro. A comparacao so faz sentido se voce ja viu a versao manual.

## 1. As mesmas ferramentas, agora com o decorator `@tool`

Reaproveitamos a logica das ferramentas do projeto anterior (calculadora segura via AST, consulta de clima). A diferenca e que, no LangChain/LangGraph, ferramentas sao declaradas com o decorator `@tool` -- ele gera automaticamente o schema JSON a partir da assinatura da funcao e da docstring, em vez de voce escrever o schema a mao (como fizemos no `TOOLS_SCHEMA` do notebook anterior).

In [ ]:
import ast
import operator as op
from langchain_core.tools import tool

_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Expressao nao suportada: {ast.dump(node)}")


@tool
def calculator(expression: str) -> str:
    """Avalia uma expressao aritmetica simples (soma, subtracao, multiplicacao, divisao, potencia).

    Args:
        expression: a expressao matematica, ex: '12 * 7'
    """
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval_node(tree.body)
        return str({"result": result})
    except Exception as e:
        return str({"error": f"Nao foi possivel avaliar \'{expression}\': {e}"})


_FAKE_WEATHER_DB = {
    "sao paulo": {"temp_c": 22, "condicao": "nublado"},
    "rio de janeiro": {"temp_c": 28, "condicao": "ensolarado"},
    "belo horizonte": {"temp_c": 19, "condicao": "chuvoso"},
}

@tool
def get_weather(city: str) -> str:
    """Retorna a temperatura e condicao do tempo atual de uma cidade.

    Args:
        city: nome da cidade, ex: 'Sao Paulo'
    """
    key = city.strip().lower()
    if key not in _FAKE_WEATHER_DB:
        return str({"error": f"cidade \'{city}\' nao encontrada na base de dados"})
    return str(_FAKE_WEATHER_DB[key])


tools = [calculator, get_weather]

# repare: nao escrevemos nenhum JSON Schema a mao -- o @tool gera a partir
# da assinatura (city: str) e da docstring. Confira o schema gerado:
print(calculator.name, "->", calculator.description)
print(calculator.args)


## 2. O estado do grafo

Em LangGraph, o "historico de mensagens" que gerenciamos manualmente no notebook anterior (a lista `messages`, com `.append()` a cada passo) vira um **estado tipado**. A parte interessante e o `Annotated[list, add_messages]`: isso diz ao LangGraph *como* mesclar novas mensagens no estado a cada passo do grafo (anexar, em vez de substituir) -- no harness manual, essa logica de "append" estava espalhada dentro do loop; aqui e declarada uma vez, no tipo do estado.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


## 3. Os nos do grafo

Um agente LangGraph e um grafo com (pelo menos) dois nos:

- **`agent`**: chama o LLM com as ferramentas disponiveis (`bind_tools`) e devolve a resposta
- **`tools`**: executa as ferramentas pedidas pelo LLM

Repare que **nao escrevemos o dispatcher manualmente** -- `ToolNode` (importado de `langgraph.prebuilt`) ja faz exatamente o que o nosso `execute_tool()` fazia no notebook anterior: olha os `tool_calls` da ultima mensagem, executa a funcao correta, e empacota o resultado no formato esperado. Tambem trata erros de execucao de ferramenta automaticamente (devolve o erro como conteudo da mensagem, em vez de lancar excecao).

In [ ]:
from langchain_anthropic import ChatAnthropic
from langgraph.prebuilt import ToolNode

llm = ChatAnthropic(model="claude-sonnet-4-6")
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: AgentState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tool_node = ToolNode(tools)


## 4. A aresta condicional (equivalente ao `if response.stop_reason != "tool_use"` do harness manual)

No notebook anterior, a decisao "o modelo pediu uma ferramenta ou ja terminou?" era um `if` dentro do loop `while`. Em LangGraph, essa decisao vira uma **aresta condicional**: uma funcao que olha o estado e devolve o nome do proximo no a executar.

In [ ]:
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"


## 5. Montar o grafo

Aqui e onde o "loop" do harness manual vira, literalmente, um **ciclo no grafo**: `tools -> agent -> (tools ou fim) -> agent -> ...`. O `max_steps` que escrevemos a mao vira o parametro `recursion_limit`, passado na hora de rodar (nao na hora de montar o grafo).

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
graph.add_edge("tools", "agent")  # depois de rodar a ferramenta, volta para o agente decidir o proximo passo

app = graph.compile()


**Bonus**: LangGraph consegue desenhar o proprio grafo automaticamente -- util para visualizar fluxos mais complexos do que este exemplo simples.

In [ ]:
# requer o pacote 'grandalf' ou renderizar via mermaid; em notebooks Jupyter normalmente:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))


## 6. Rodar o agente

A chamada de execucao (`app.invoke`) substitui inteiramente o nosso `run_agent()` manual -- o loop, o limite de passos (via `recursion_limit`) e o parsing de tool calls ja estao dentro do grafo compilado.

In [ ]:
result = app.invoke(
    {"messages": [("user", "Qual e o clima em Sao Paulo, e quanto e 15% desse valor de temperatura em graus?")]},
    config={"recursion_limit": 10},  # equivalente ao max_steps do harness manual
)

for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content}")


## 7. Comparacao direta: harness manual vs. LangGraph

| Responsabilidade | Harness manual (notebook anterior) | LangGraph |
|---|---|---|
| Gerar schema da ferramenta | Escrito a mao em `TOOLS_SCHEMA` (JSON Schema) | Gerado automaticamente pelo `@tool` a partir da assinatura e docstring |
| Loop de execucao | `while steps < max_steps:` escrito a mao | Ciclo implicito no grafo (`tools -> agent -> ...`) |
| Limite de passos | Variavel `max_steps` verificada manualmente | Parametro `recursion_limit` no `invoke()` |
| Decidir se continua ou termina | `if response.stop_reason != "tool_use":` | Funcao de aresta condicional (`should_continue`) |
| Executar a ferramenta certa | Dicionario `TOOL_FUNCTIONS` + `execute_tool()` escrito a mao | `ToolNode(tools)` pronto, importado do framework |
| Tratamento de erro de ferramenta | `try/except` escrito a mao dentro de `execute_tool` | `ToolNode` ja captura excecoes e devolve como conteudo de erro |
| Gerenciar o historico de mensagens | `messages.append(...)` manual a cada passo | `Annotated[list, add_messages]` no estado -- mesclagem automatica |
| Memoria entre execucoes (nao coberto no notebook anterior) | Precisaria ser implementado do zero | `checkpointer` (ex: `MemorySaver`) plugavel diretamente no grafo |
| Streaming de passos intermediarios | Precisaria ser implementado do zero | `app.stream(...)` pronto |
| Visualizar o fluxo do agente | Nao existe (e so codigo Python) | `app.get_graph().draw_mermaid_png()` |

**O que isso revela**: LangGraph nao adiciona uma capacidade fundamentalmente nova ao que voce ja tinha no harness manual -- ele **empacota padroes recorrentes** (loop de tool-use, roteamento condicional, merge de historico) em componentes reutilizaveis, alguns dos quais (checkpointer, streaming, visualizacao) exigiriam trabalho de infraestrutura considerável se voce fosse escrever a mao para um agente de producao.

## Notas finais

- **Quando vale a pena usar o framework**: quando voce precisa de recursos que exigiriam bastante trabalho de infraestrutura para replicar a mao -- memoria persistente entre sessoes (`checkpointer`), streaming de passos intermediarios para uma interface, grafos com multiplos caminhos condicionais complexos (nao so um loop simples de tool-use)
- **Quando o harness manual ainda faz sentido**: prototipos rapidos, agentes simples de passo unico ou poucos passos, ou quando voce precisa de controle fino sobre um comportamento que o framework nao expõe facilmente
- **Um ponto de atencao real**: como o framework abstrai o loop, um bug de comportamento (ex: o agente ficando preso chamando a mesma ferramenta) fica *mais dificil de debugar* do que no harness manual, onde cada linha do loop esta a vista. Isso reforca por que vale a pena ter construido a versao manual primeiro -- agora voce sabe o que procurar quando algo em LangGraph nao se comporta como esperado
- **Proximo projeto da fase**: agente com multiplas ferramentas (incluindo RAG da Fase 4) e memoria entre perguntas -- LangGraph com `checkpointer` e um bom ponto de partida natural para isso
